## Model Training

In [ ]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.datasets import Food101
from PIL import Image

## Load Data


In [ ]:
NUM_CLASSES = 101
IMAGE_SIZE = 224

# Imagenet normaliztion stats - required since we are using weights trained on ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

test_transforms = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE * 1.14)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])


In [ ]:
train_dataset = Food101(root="../data/food-101/", split='train', transform=train_transforms, download=True)
test_dataset = Food101(root="../data/food-101/", split='test', transform=test_transforms, download=True)

## Train-Validation Split

In this step the training set is spilt into two smaller subsets. The first one is used to train the model and the second is used to evaluate the performance of the model during training and for hyperparameter tuning.

The provided test set remains hidden during the training, evaluation and hyperparameter tuning process and is used as a final unbiased evaluation of the selected model.

In [ ]:
dataset = train_dataset
indices = np.arange(len(dataset))

# Random seed for reproducibility
np.random.seed(42)

# Shuffle the indeces
np.random.shuffle(indices)

# 20/80 split - 80% training data, 20% validation data
# The idea here is to make the validation data small enough to leave enough samples for training
# but not too small or it won't be a representative sample
split = int(0.2 * len(dataset))

val_indeces = indices[:split]
train_indeces = indices[split:]

val_sampler = SubsetRandomSampler(val_indeces)
train_sampler = SubsetRandomSampler(train_indeces)

train_loader = DataLoader(dataset, batch_size=64, sampler=train_sampler)
val_loader = DataLoader(dataset, batch_size=64, sampler=val_sampler)